# Khai báo và đọc dữ liệu

In [2]:
import pandas as pd
import numpy as np

RAW = '../datas/raw/binh_duong_real_estates.csv'
OUT_CLEAN = '../datas/processed/cleaned_binh_duong_real_estates.csv'
OUT_OUTLIER = '../datas/processed/outliers_binh_duong_real_estates.csv'
OUT_SEGMENT = '../datas/processed/segment_thresholds.csv'

UNKNOWN = 'Không xác định'
NO_PROJECT = 'Không thuộc dự án'
BIZ_COLS = ['name', 'price', 'area', 'ward_name', 'district_name']

df = pd.read_csv(RAW, encoding='utf-8-sig')
log = []

def track(step, before, after):
    log.append({'Bước': step, 'Số dòng loại': before - after, 'Số dòng còn lại': after})

log.append({'Bước': 'Sau lọc Bình Dương', 'Số dòng loại': 0, 'Số dòng còn lại': len(df)})

# R1 và R2, bỏ cột kỹ thuật và tách ngày giờ

In [4]:
# R1: bỏ 4 cột thiếu trên 80%
df = df.drop(columns=['house_depth', 'floor_count', 'road_width', 'balcony_direction'], errors='ignore')

# R2: đọc thời điểm đăng theo chuẩn ISO 8601, tách thành ngày và giờ
dt = pd.to_datetime(df['published_at'], format='ISO8601', errors='coerce')
df['full_date'] = dt.dt.strftime('%Y-%m-%d')
df['post_hour'] = dt.dt.hour.astype('Int64')
df = df.drop(columns=['published_at'])

# R3, ép kiểu số và giữ NULL

In [5]:
# R3: chuyển cột số về kiểu số, giá trị lỗi để trống
NUM_COLS = ['price', 'area', 'frontage_width', 'bedroom_count', 'bathroom_count']
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# R4 và R5, chuẩn hóa văn bản và nhãn

In [6]:
# R4: cắt khoảng trắng đầu và cuối
TEXT_COLS = ['name', 'district_name', 'ward_name', 'street_name', 'project_name', 'house_direction']
for c in TEXT_COLS:
    df[c] = df[c].astype('string').str.strip()

In [7]:
import numpy as np

UNKNOWN = 'Không xác định'
NO_PROJECT = 'Không thuộc dự án'

# Nhãn loại hình thực tế trong dữ liệu có thể nằm trong nhóm nhà lẻ
LANDED = ['Nhà', 'Biệt thự/Nhà liền kề', 'Đất']

# R5a: nhãn không xác định cho địa điểm
for c in ['ward_name', 'street_name']:
    df[c] = df[c].replace(['', 'Chưa xác định'], UNKNOWN).fillna(UNKNOWN)

# R5b: hướng nhà, thay nhãn trống và bỏ dấu gạch
df['house_direction'] = (
    df['house_direction']
    .fillna(UNKNOWN)
    .replace(['', 'Chưa xác định'], UNKNOWN)
    .str.replace(' - ', ' ', regex=False)
)

# R5c: tên dự án
# Nhà lẻ thiếu tên dự án -> NO_PROJECT
# Căn hộ, shophouse thiếu tên dự án -> UNKNOWN
proj = df['project_name'].fillna('').str.strip()
no_proj = proj.isin(['', 'Chưa xác định'])

df['project_name'] = np.select(
    [no_proj & df['property_type_name'].isin(LANDED), no_proj],
    [NO_PROJECT, UNKNOWN],
    default=proj
)

# R6 đến R9, loại dòng

In [8]:
# R6: thiếu giá
n0 = len(df)
df = df.dropna(subset=['price'])
track('R6 Thiếu giá', n0, len(df))

# R7: trùng tuyệt đối
n0 = len(df)
df = df.drop_duplicates()
track('R7 Trùng tuyệt đối', n0, len(df))

# R8: trùng nghiệp vụ, giữ bản ghi đầu tiên
n0 = len(df)
df = df.drop_duplicates(subset=BIZ_COLS, keep='first')
track('R8 Trùng nghiệp vụ', n0, len(df))

# R9: diện tích dưới 10 m2 (giữ NaN, không loại dòng thiếu diện tích)
n0 = len(df)
df = df[~(df['area'] < 10)]
track('R9 Diện tích dưới 10 m2', n0, len(df))

# R10, tách tổng giá cực lớn

In [9]:
# R10: tổng giá trên 500 tỷ đồng, tách sang file ngoại lệ
mask_big = df['price'] > 5e11
outlier_file = df[mask_big].copy()
outlier_file['ly_do'] = 'Tổng giá trên 500 tỷ đồng'

n0 = len(df)
df = df[~mask_big]
track('R10 Tổng giá trên 500 tỷ', n0, len(df))

# R11, gắn cờ đơn giá ngoại lệ

In [10]:
# R11: đơn giá ngoài phân vị 0,5% và 99,5%, gắn cờ và giữ trong fact
df['price_per_m2'] = df['price'] / df['area']

lo, hi = df['price_per_m2'].quantile([0.005, 0.995])
df['is_outlier'] = ((df['price_per_m2'] < lo) | (df['price_per_m2'] > hi)).astype(int)

print(f"Ngưỡng: {lo:,.0f} đến {hi:,.0f} VNĐ/m2")
print("Số dòng gắn cờ:", df['is_outlier'].sum())

Ngưỡng: 692,308 đến 129,250,394 VNĐ/m2
Số dòng gắn cờ: 360


# R12, gán phân khúc

In [11]:
# R12: phân khúc theo tứ phân vị của các dòng không ngoại lệ
ref = df.loc[df['is_outlier'] == 0, 'price_per_m2']
q1, q2, q3 = ref.quantile([0.25, 0.5, 0.75])

bins = [-np.inf, q1, q2, q3, np.inf]
df['segment_key'] = (pd.cut(df['price_per_m2'], bins=bins, labels=[1, 2, 3, 4], include_lowest=True)
                     .astype('float').fillna(-1).astype(int))

segment = pd.DataFrame({
    'segment_key': [1, 2, 3, 4, -1],
    'segment_name': ['Phân khúc thấp', 'Trung bình', 'Cao cấp', 'Rất cao', UNKNOWN],
    'min_price_per_m2': [ref.min(), q1, q2, q3, None],
    'max_price_per_m2': [q1, q2, q3, ref.max(), None],
    'sort_order': [1, 2, 3, 4, 0]
})

# Xuất file và kiểm tra

In [12]:
clean = df.drop(columns=['price_per_m2'])
outlier_out = outlier_file.drop(columns=['published_at'], errors='ignore')

# Làm tròn theo độ chính xác của cột đích trong Stg_Listing,
# tránh sai số biểu diễn của số thực (ví dụ 5.5600000000000005)
clean['frontage_width'] = clean['frontage_width'].round(2)   # DECIMAL(10,2)
clean['area'] = clean['area'].round(3)                       # DECIMAL(12,3)

clean.to_csv(OUT_CLEAN, index=False, encoding='utf-8-sig')
outlier_out.to_csv(OUT_OUTLIER, index=False, encoding='utf-8-sig')
segment.to_csv(OUT_SEGMENT, index=False, encoding='utf-8-sig')

log.append({'Bước': 'Dữ liệu sạch cuối cùng', 'Số dòng loại': 0, 'Số dòng còn lại': len(clean)})
display(pd.DataFrame(log))

# Kiểm tra sau khi xuất
print("Tháng trong dữ liệu sạch:", sorted(pd.to_datetime(clean['full_date']).dt.month.unique()))
print("Số dòng sạch:", len(clean))
assert len(clean) == 36303, "Số dòng không khớp với nhật ký R10"
assert 'price_per_m2' not in clean.columns

print("bedroom_count = 0 (giá trị thật):", (clean['bedroom_count'] == 0).sum())
print("bedroom_count NULL (giữ nguyên):", clean['bedroom_count'].isna().sum())
print("Số dòng is_outlier = 1:", clean['is_outlier'].sum())
print("Phân bố phân khúc:", clean['segment_key'].value_counts().sort_index().to_dict())
print("frontage_width tại dòng 27216:", clean.loc[27216, 'frontage_width'])

,Bước,Số dòng loại,Số dòng còn lại
0,Sau lọc Bình Dương,0,42783
1,R6 Thiếu giá,3324,39459
2,R7 Trùng tuyệt đối,33,39426
3,R8 Trùng nghiệp vụ,3104,36322
4,R9 Diện tích dưới 10 m2,2,36320
5,R10 Tổng giá trên 500 tỷ,17,36303
6,Dữ liệu sạch cuối cùng,0,36303


Tháng trong dữ liệu sạch: [np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
Số dòng sạch: 36303
bedroom_count = 0 (giá trị thật): 0
bedroom_count NULL (giữ nguyên): 15323
Số dòng is_outlier = 1: 360
Phân bố phân khúc: {1: 9164, 2: 9056, 3: 8936, 4: 9147}
frontage_width tại dòng 27216: nan
